# Swin-Base Fine-tuning on ImageNet-100

**Model:** `microsoft/swin-base-patch4-window7-224`  
**Dataset:** ImageNet-100 (100-class subset of ImageNet-1K)  
**Framework:** PyTorch + HuggingFace Transformers  

**Training strategy:**
- Phase 1 (epochs 1-5): Freeze backbone, train 100-class head only
- Phase 2 (epochs 6-20): Unfreeze all layers, full fine-tuning at LR/10
- Mixed-precision (AMP) via `torch.cuda.amp`

**Outputs:**
- `Swin_Training/checkpoints/swin_base_imagenet100_best.pt`
- `Swin_Training/checkpoints/swin_base_imagenet100_final.pt`
- `Swin_Training/clean_baselines/swin_base_clean.json`
- `Swin_Training/logs/swin_training_<timestamp>.log`

In [ ]:
import sys, os, json, logging
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import SwinForImageClassification, AutoImageProcessor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths ──
NOTEBOOK_DIR = Path('.').resolve()              # VIT/
PROJECT_ROOT = NOTEBOOK_DIR.parent             # project root
sys.path.insert(0, str(NOTEBOOK_DIR / 'src'))  # swin_utils

from swin_utils import (
    seed_everything, get_device, ImageNet100Dataset, load_swin_config,
    evaluate_clean,
)

# ── Logging ──
LOG_DIR = PROJECT_ROOT / 'Swin_Training' / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path  = LOG_DIR / f'swin_training_{timestamp}.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger('swin_training')
logger.info(f'Log file: {log_path}')

In [ ]:
# ── Configuration ──
SEED           = 42
CHECKPOINT     = 'microsoft/swin-base-patch4-window7-224'
NUM_CLASSES    = 100
EPOCHS         = 20
FROZEN_EPOCHS  = 5      # Phase 1: head-only
BATCH_SIZE     = 32
LR             = 1e-4
MIN_LR         = 1e-6
NUM_WORKERS    = 4
USE_AMP        = True   # Mixed precision

DATA_DIR     = PROJECT_ROOT / 'ImageNet100_Training' / 'data'
MAPPING_FILE = PROJECT_ROOT / 'ImageNet100_Training' / 'class_mapping.json'
OUT_DIR      = PROJECT_ROOT / 'Swin_Training'
CKPT_DIR     = OUT_DIR / 'checkpoints'
BASELINES_DIR = OUT_DIR / 'clean_baselines'
FIG_DIR      = OUT_DIR / 'figures'

for d in [CKPT_DIR, BASELINES_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
USE_AMP = USE_AMP and device.type == 'cuda'

logger.info(f'Device: {device}  AMP: {USE_AMP}')
logger.info(f'Data dir: {DATA_DIR}')

In [ ]:
# ── Verify data directory ──
assert DATA_DIR.exists(), (
    f'Data directory not found: {DATA_DIR}\n'
    'Run scripts/setup_imagenet100.py first.'
)
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    if split_dir.exists():
        n_classes = len([d for d in split_dir.iterdir() if d.is_dir()])
        n_images  = sum(1 for _ in split_dir.rglob('*.JPEG'))
        logger.info(f'  {split}: {n_classes} classes, {n_images} images')
    else:
        logger.warning(f'  {split} split not found!')

# ── Load class mapping ──
with open(MAPPING_FILE) as f:
    class_mapping = json.load(f)

class_names = [
    class_mapping[s]['human_name']
    for s in sorted(class_mapping, key=lambda x: class_mapping[x]['index'])
]
logger.info(f'Classes: {NUM_CLASSES}  Example: {class_names[:5]}')

In [ ]:
# ── Load processor & build DataLoaders ──
logger.info('Loading processor and datasets...')
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)

train_ds = ImageNet100Dataset(DATA_DIR, split='train', processor=processor)
val_ds   = ImageNet100Dataset(DATA_DIR, split='val',   processor=processor)
test_ds  = ImageNet100Dataset(DATA_DIR, split='test',  processor=processor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

logger.info(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
# ── Build model ──
logger.info(f'Loading {CHECKPOINT}...')
model = SwinForImageClassification.from_pretrained(CHECKPOINT)

# Replace 1000-class head with 100-class head
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
nn.init.trunc_normal_(model.classifier.weight, std=0.02)
nn.init.zeros_(model.classifier.bias)

model = model.to(device)
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f'Parameters: {total_params:,} total | {trainable_params:,} trainable')

In [ ]:
# ── Training helpers ──
def train_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_i, (pv, labels) in enumerate(loader):
        pv     = pv.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            out  = model(pixel_values=pv)
            loss = F.cross_entropy(out.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * pv.size(0)
        correct    += (out.logits.argmax(-1) == labels).sum().item()
        total      += pv.size(0)

        if (batch_i + 1) % 50 == 0:
            print(f'  Epoch {epoch} [{batch_i+1}/{len(loader)}] '
                  f'loss={total_loss/total:.4f} acc={correct/total:.3f}', end='\r')
    print()
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total, losses = 0, 0, []
    for pv, labels in loader:
        pv     = pv.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out    = model(pixel_values=pv)
        losses.append(F.cross_entropy(out.logits, labels).item())
        correct += (out.logits.argmax(-1) == labels).sum().item()
        total   += pv.size(0)
    return float(np.mean(losses)), correct / total


def save_checkpoint(model, optimizer, scaler, epoch, val_acc, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'epoch': epoch, 'val_acc': val_acc,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
    }, path)
    logger.info(f'  Saved checkpoint: {path.name}  (val_acc={val_acc:.4f})')

In [ ]:
# ── Phase 1: Head-only training ──
best_val_acc = 0.0
best_ckpt    = CKPT_DIR / 'swin_base_imagenet100_best.pt'
scaler       = GradScaler(enabled=USE_AMP)
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

logger.info(f'\n[Phase 1] Head-only training for {FROZEN_EPOCHS} epochs...')
for name, param in model.named_parameters():
    param.requires_grad = 'classifier' in name
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f'  Frozen backbone. Trainable params: {trainable:,} (head only)')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.05
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=FROZEN_EPOCHS, eta_min=MIN_LR
)

for epoch in range(1, FROZEN_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scaler, epoch)
    vl_loss, vl_acc = evaluate(model, val_loader)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    logger.info(f'  Epoch {epoch:2d}/{FROZEN_EPOCHS} | '
                f'train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
                f'val loss={vl_loss:.4f} acc={vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        save_checkpoint(model, optimizer, scaler, epoch, vl_acc, best_ckpt)

In [ ]:
# ── Phase 2: Full fine-tuning ──
remaining = EPOCHS - FROZEN_EPOCHS
logger.info(f'\n[Phase 2] Full fine-tuning for {remaining} epochs...')
for param in model.parameters():
    param.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f'  Unfrozen. Trainable params: {trainable:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR / 10, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=remaining, eta_min=MIN_LR
)

for epoch in range(FROZEN_EPOCHS + 1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scaler, epoch)
    vl_loss, vl_acc = evaluate(model, val_loader)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    logger.info(f'  Epoch {epoch:2d}/{EPOCHS} | '
                f'train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
                f'val loss={vl_loss:.4f} acc={vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        save_checkpoint(model, optimizer, scaler, epoch, vl_acc, best_ckpt)

# Save final checkpoint
final_ckpt = CKPT_DIR / 'swin_base_imagenet100_final.pt'
torch.save(model.state_dict(), final_ckpt)
logger.info(f'Final checkpoint saved: {final_ckpt}')
logger.info(f'Best val accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)
axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].axvline(FROZEN_EPOCHS + 0.5, color='gray', linestyle='--', label='Phase 2 start')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Val')
axes[1].axvline(FROZEN_EPOCHS + 0.5, color='gray', linestyle='--', label='Phase 2 start')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Swin-Base Training on ImageNet-100', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
logger.info(f'Training curves saved.')

In [ ]:
# ── Evaluate best checkpoint on test set ──
logger.info('\nLoading best checkpoint for test evaluation...')
best_state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(best_state['model'])

test_results = evaluate_clean(model, test_loader, device, num_classes=NUM_CLASSES)
logger.info(f'Test accuracy:    {test_results["accuracy"]:.4f}')
logger.info(f'Mean confidence:  {test_results["mean_confidence"]:.4f}')
logger.info(f'Total samples:    {test_results["total_samples"]}')

In [ ]:
# ── Save clean baseline JSON ──
per_class_acc = {
    class_names[i]: round(v, 4)
    for i, v in test_results['per_class_accuracy'].items()
}

baseline = {
    'model':           'swin-base-patch4-window7-224',
    'dataset':         'ImageNet-100',
    'split':           'test',
    'accuracy':        round(test_results['accuracy'], 4),
    'mean_confidence': round(test_results['mean_confidence'], 4),
    'total_samples':   test_results['total_samples'],
    'per_class_accuracy': per_class_acc,
    'timestamp':       datetime.now().isoformat(),
}

baseline_path = BASELINES_DIR / 'swin_base_clean.json'
with open(baseline_path, 'w') as f:
    json.dump(baseline, f, indent=2)
logger.info(f'Clean baseline saved: {baseline_path}')

In [ ]:
# ── Per-class accuracy analysis ──
accs_sorted = sorted(per_class_acc.items(), key=lambda x: x[1])
bottom10 = accs_sorted[:10]
top10    = accs_sorted[-10:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in [
    (axes[0], bottom10, 'Bottom-10 Classes', 'tomato'),
    (axes[1], top10,    'Top-10 Classes',    'steelblue'),
]:
    names = [n[:18] for n, _ in data]
    vals  = [v for _, v in data]
    ax.barh(names, vals, color=color, edgecolor='white')
    ax.set_xlim(0, 1); ax.set_xlabel('Accuracy')
    ax.set_title(title); ax.grid(axis='x', alpha=0.3)

plt.suptitle(f'Per-class Accuracy — Swin-Base ImageNet-100 '
             f'(mean={test_results["accuracy"]:.3f})')
plt.tight_layout()
fig.savefig(FIG_DIR / 'per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

print(f'\nBottom-5 classes:')
for name, acc in bottom10[:5]:
    print(f'  {name}: {acc:.3f}')
print(f'\nTop-5 classes:')
for name, acc in top10[-5:]:
    print(f'  {name}: {acc:.3f}')

In [ ]:
# ── Summary ──
print('=' * 60)
print('  SWIN-BASE TRAINING COMPLETE')
print('=' * 60)
print(f'  Best val accuracy  : {best_val_acc:.4f}')
print(f'  Test accuracy      : {test_results["accuracy"]:.4f}')
print(f'  Mean confidence    : {test_results["mean_confidence"]:.4f}')
print(f'  Best checkpoint    : {best_ckpt}')
print(f'  Final checkpoint   : {final_ckpt}')
print(f'  Clean baseline     : {baseline_path}')
print(f'  Log file           : {log_path}')
print('=' * 60)